In [ ]:
from myosuite.utils import gym
import skvideo.io
import numpy as np
import matplotlib.pyplot as plt
import pickle
import time
import os

os.environ['MUJOCO_GL'] = 'egl'

import mujoco

In [ ]:
from IPython.display import HTML
from base64 import b64encode
 
def show_video(video_path, video_width=400):
   
  video_file = open(video_path, "r+b").read()
 
  video_url = f"data:video/mp4;base64,{b64encode(video_file).decode()}"
  return HTML(f"""<video autoplay width={video_width} controls><source src="{video_url}"></video>""")


import PIL.Image, PIL.ImageDraw, PIL.ImageFont

def add_text_to_frame(frame, text, font="dejavu/DejaVuSans.ttf", pos=(20, 20), color=(255, 0, 0), fontsize=12):
    if isinstance(frame, np.ndarray):
        frame = PIL.Image.fromarray(frame)
    
    draw = PIL.ImageDraw.Draw(frame)
    draw.text(pos, text, fill=color, font=PIL.ImageFont.truetype(font, fontsize))
    return frame

# Fatigue Modeling

MyoSuite includes a fatigue model based on the (modified) ["Three Compartment Controller (3CC-r)" model](https://doi.org/10.1016/j.jbiomech.2018.06.005). \
The implementation is based on the *CumulativeFatigue* model included in the ["User-in-the-Box" framework](https://github.com/aikkala/user-in-the-box/blob/main/uitb/bm_models/effort_models.py).
For details on the dynamics of the 3CC-r model, we refer the interested readers to the papers from [Looft et al.](https://doi.org/10.1016/j.jbiomech.2018.06.005) and [Cheema et al.](https://doi.org/10.1145/3313831.3376701).

Crucially, the 3CC-r model is implemented on a muscle level, i.e., fatigue is computed for each muscle individually rather than for a single (shoulder) joint. \
While originally built and tested for models of the arm, elbow and hand, it can also be used with models of the lower extremity, e.g., the MyoLeg model.

## Use the Fatigue Model

To model fatigue, load the desired MyoSuite environment in its "Fati" variant:

In [ ]:
envFatigue = gym.make('myoFatiElbowPose1D6MRandom-v0', normalize_act=False)

## for comparison
env = gym.make('myoElbowPose1D6MRandom-v0', normalize_act=False)

This adds the "muscle_fatigue" attribute, which entails the current fatigue state of each muscle:

In [ ]:
envFatigue.muscle_fatigue.MF   #percentage of fatigued motor units for each muscle
envFatigue.muscle_fatigue.MR   #percentage of resting motor units for each muscle
envFatigue.muscle_fatigue.MA   #percentage of active motor units for each muscle

The fatigue/recovery constants F and R as well as the recovery multiplier r, which determines how much faster motor units recover during rest periods (i.e., when less motor units are required than are currently active), can be set using the following methods:

In [ ]:
envFatigue.muscle_fatigue.set_RecoveryMultiplier(10)
envFatigue.muscle_fatigue.set_RecoveryCoefficient(0.0022)
envFatigue.muscle_fatigue.set_FatigueCoefficient(0.0146)

envFatigue.muscle_fatigue.r, envFatigue.muscle_fatigue.R, envFatigue.muscle_fatigue.F

**Note:** The parameters F and R (and in particular their ratio, which defines the percentage of active motor units in a totally fatigued state) generally depend on the joints and muscles, and thus need to be manually chosen for each model!

The muscle force development/relaxation factors LD and LR are automatically computed for each muscle based on its time activation and deactivation constants.

#### Initial Fatigue States

By default, the simulation starts in the default "non-fatigued" muscle state with 100% resting motor units.

In [ ]:
envFatigue.reset()
envFatigue.muscle_fatigue.MF, envFatigue.muscle_fatigue.MR, envFatigue.muscle_fatigue.MA

To reset to a randomly chosen distribution of fatigued, resting and active motor units per muscle, the following command can be used:

In [ ]:
envFatigue.set_fatigue_reset_random(True)

envFatigue.reset()
envFatigue.muscle_fatigue.MF, envFatigue.muscle_fatigue.MR, envFatigue.muscle_fatigue.MA

In the fatigue variant, the simulation applies the "currently available" muscle controls as defined by the fatigue state and dynamics, rather than the intended muscle control signals a:

In [ ]:
envFatigue.set_fatigue_reset_random(False)
a = np.zeros(envFatigue.sim.model.nu,)
a[0] = 1

envFatigue.reset()
for i in range(10):
    next_o, r, done, *_, ifo = envFatigue.step(a) # take an action

# Comparison: without fatigue
env.reset()
for i in range(10):
    next_o, r, done, *_, ifo = env.step(a) # take an action

env.last_ctrl, envFatigue.last_ctrl, envFatigue.muscle_fatigue.MF

This allows to predict how fatigue evolves over time:

In [ ]:
env.reset()
envFatigue.reset()
data_store = []
data_store_f = []
for i in range(7*3): # 7 batches of 3 episodes, with 2 episodes of maximum muscle controls for some muscles followed by a resting episode (i.e., zero muscle controls) in each batch
    a = np.zeros(env.sim.model.nu,)
    if i%3!=2:
        a[3:]=1
    else:
        a[:]=0
    
    for _ in range(500): # 500 samples (=10s) for each episode
        next_o, r, done, *_, ifo = env.step(a) # take an action
        next_f_o, r_f, done_F, *_, ifo_f = envFatigue.step(a) # take an action
                    
        data_store.append({"action":a.copy(), 
                            "jpos":env.sim.data.qpos.copy(), 
                            "mlen":env.sim.data.actuator_length.copy(), 
                            "act":env.sim.data.act.copy()})
        data_store_f.append({"action":a.copy(), 
                            "jpos":envFatigue.sim.data.qpos.copy(), 
                            "mlen":envFatigue.sim.data.actuator_length.copy(),
                            "MF":envFatigue.muscle_fatigue.MF.copy(),
                            "MR":envFatigue.muscle_fatigue.MR.copy(),
                            "MA":envFatigue.muscle_fatigue.MA.copy(), 
                            "act":envFatigue.sim.data.act.copy()})

env.close()
envFatigue.close()

muscle_names = [env.sim.model.id2name(i, "actuator") for i in range(env.sim.model.nu) if env.sim.model.actuator_dyntype[i] == mujoco.mjtDyn.mjDYN_MUSCLE]
muscle_id = -1

plt.figure(figsize=(12, 6))
plt.subplot(221)
plt.plot(env.dt*np.arange(len(data_store)), np.array([d['act'][muscle_id] for d in data_store]), label="Normal model/Desired activations")
plt.plot(env.dt*np.arange(len(data_store)), np.array([d['act'][muscle_id] for d in data_store_f]), label='Fatigued model')
plt.legend()
plt.title(f'Muscle activations over time ({muscle_names[muscle_id]})')
plt.xlabel('time (s)'),plt.ylabel('act')

plt.subplot(222)
plt.plot(env.dt*np.arange(len(data_store)), np.array([d['jpos'] for d in data_store]), label="Normal model")
plt.plot(env.dt*np.arange(len(data_store)), np.array([d['jpos'] for d in data_store_f]), label="Fatigued model")
plt.legend()
plt.title('Joint angle over time')
plt.xlabel('time (s)'),plt.ylabel('angle')

plt.subplot(223)
plt.plot(env.dt*np.arange(len(data_store)), np.array([d['mlen'][muscle_id] for d in data_store]), label="Normal model")
plt.plot(env.dt*np.arange(len(data_store)), np.array([d['mlen'][muscle_id] for d in data_store_f]), label="Fatigued model")
plt.legend()
plt.title(f'Muscle lengths over time ({muscle_names[muscle_id]})')
plt.xlabel('time (s)'),plt.ylabel('muscle length')

plt.subplot(224)
plt.plot(env.dt*np.arange(len(data_store)), np.array([d['MF'][muscle_id] for d in data_store_f]), color="tab:orange")
plt.title(f'Fatigued motor units over time ({muscle_names[muscle_id]})')
plt.xlabel('time (s)'),plt.ylabel('%MVC')

plt.tight_layout()
plt.show()

In [ ]:
env.reset()
envFatigue.reset()
data_store = []
data_store_f = []
for i in range(2*3): # 2 batches of 3 episodes, with 0.5*MVC in first and 1*MVC in second episode, followed by a resting episode (i.e., zero muscle controls) in each batch
    a = np.zeros(env.sim.model.nu,)
    if i%3==0:
        a[3:]=0.5
    elif i%3==1:
        a[3:]=1
    else:
        a[:]=0
    
    for _ in range(9000): # 9000 samples (=3 minutes) for each episode
        next_o, r, done, *_, ifo = env.step(a) # take an action
        next_f_o, r_f, done_F, *_, ifo_f = envFatigue.step(a) # take an action
                    
        data_store.append({"action":a.copy(), 
                            "jpos":env.sim.data.qpos.copy(), 
                            "mlen":env.sim.data.actuator_length.copy(), 
                            "act":env.sim.data.act.copy()})
        data_store_f.append({"action":a.copy(), 
                            "jpos":envFatigue.sim.data.qpos.copy(), 
                            "mlen":envFatigue.sim.data.actuator_length.copy(),
                            "MF":envFatigue.muscle_fatigue.MF.copy(),
                            "MR":envFatigue.muscle_fatigue.MR.copy(),
                            "MA":envFatigue.muscle_fatigue.MA.copy(),
                            "act":envFatigue.sim.data.act.copy()})

env.close()
envFatigue.close()

muscle_names = [env.sim.model.id2name(i, "actuator") for i in range(env.sim.model.nu) if env.sim.model.actuator_dyntype[i] == mujoco.mjtDyn.mjDYN_MUSCLE]
muscle_id = -1

plt.figure(figsize=(12, 6))
plt.subplot(221)
plt.plot(env.dt*np.arange(len(data_store)), np.array([d['act'][muscle_id] for d in data_store]), label="Normal model/Desired activations")
plt.plot(env.dt*np.arange(len(data_store)), np.array([d['act'][muscle_id] for d in data_store_f]), label='Fatigued model')
plt.legend()
plt.title(f'Muscle activations over time ({muscle_names[muscle_id]})')
plt.xlabel('time (s)'),plt.ylabel('act')

plt.subplot(222)
plt.plot(env.dt*np.arange(len(data_store)), np.array([d['jpos'] for d in data_store]), label="Normal model")
plt.plot(env.dt*np.arange(len(data_store)), np.array([d['jpos'] for d in data_store_f]), label="Fatigued model")
plt.legend()
plt.title('Joint angle over time')
plt.xlabel('time (s)'),plt.ylabel('angle')

plt.subplot(223)
plt.plot(env.dt*np.arange(len(data_store)), np.array([d['mlen'][muscle_id] for d in data_store]), label="Normal model")
plt.plot(env.dt*np.arange(len(data_store)), np.array([d['mlen'][muscle_id] for d in data_store_f]), label="Fatigued model")
plt.legend()
plt.title(f'Muscle lengths over time ({muscle_names[muscle_id]})')
plt.xlabel('time (s)'),plt.ylabel('muscle length')

plt.subplot(224)
plt.plot(env.dt*np.arange(len(data_store)), np.array([d['MF'][muscle_id] for d in data_store_f]), color="tab:orange")
plt.title(f'Fatigued motor units over time ({muscle_names[muscle_id]})')
plt.xlabel('time (s)'),plt.ylabel('%MVC')

plt.tight_layout()
plt.show()

## Train Agents with Fatigue

**NOTE:** By default, random fatigue states are sampled at the beginning of each training episode. \
To start with a specific, fixed fatigue state, set `fatigue_reset_random=False` and define `fatigue_reset_vec` as the vector MF of fatigued motor units per muscle.

Best practice is to create a new  of the desired environment, i.e., calling `register_env_variant()` with
`variants={'muscle_condition': 'fatigue',
            'fatigue_reset_vec': np.array([0., 0., 0.]),
            'fatigue_reset_random': False}`.


In [ ]:
from mjrl.utils.gym_env import GymEnv
from mjrl.policies.gaussian_mlp import MLP
from mjrl.baselines.mlp_baseline import MLPBaseline
from mjrl.algos.npg_cg import NPG
from mjrl.utils.train_agent import train_agent

In [ ]:
env_name = "myoFatiElbowPose1D6MRandom-v0"

env = gym.make(env_name)
env.set_fatigue_reset_random(True)
env.reset()

policy_size = (32, 32)
vf_hidden_size = (128, 128)
seed = 123
rl_step_size = 0.1
e = GymEnv(env)

policy = MLP(e.spec, hidden_sizes=policy_size, seed=seed, init_log_std=-0.25, min_log_std=-1.0)

baseline = MLPBaseline(e.spec, reg_coef=1e-3, batch_size=64, hidden_sizes=vf_hidden_size, \
                    epochs=2, use_gpu=True, learn_rate=1e-3)

agent = NPG(e, policy, baseline, normalized_step_size=rl_step_size, \
            seed=seed, save_logs=True)

print("========================================")
print("Starting policy learning")
print("========================================")

train_agent(job_name=f'./{env_name}',
            agent=agent,
            seed=seed,
            niter=200000,
            gamma=0.995,
            gae_lambda=0.97,
            num_cpu=8,
            sample_mode="trajectories",
            num_traj=96,
            num_samples=0,
            save_freq=500,
            evaluation_rollouts=10)

print("========================================")
print("Job Finished.") 
print("========================================")

## Simulate and Evaluate Trained Agents

In the following, we evaluate the latest policy trained for a given fatigue environment (and for comparison also the policy trained in the respective non-fatigue environment).

To this end, we simulate several episodes per policy, with fatigue accumulating across episodes, starting in the default zero fatigue state. \
Videos of the first and the last few episodes are generated, and simulation data is logged (and later visualised) for all episodes.

### Example 1: Fatigue

In [ ]:
env_name = "myoFatiElbowPose1D6MRandom-v0"

GENERATE_VIDEO = True
GENERATE_VIDEO_EPS = 4  #number of episodes that are rendered BOTH at the beginning (i.e., without fatigue) and at the end (i.e., with fatigue)

STORE_DATA = True  #store collected data from evaluation run in .npy file
n_eps = 250

###################################

env = gym.make(env_name)

policy = f"{env_name}/iterations/best_policy.pickle"
pi = pickle.load(open(policy, 'rb'))

env.set_fatigue_reset_random(False)
env.reset(fatigue_reset=True)  #ensure that fatigue is reset before the simulation starts

env.sim.model.cam_poscom0[0]= np.array([-1.3955, -0.3287,  0.6579])

data_store = []
if GENERATE_VIDEO:
    frames = []

env.env.target_jnt_value = env.target_jnt_range[:, 1]
env.env.target_type = 'fixed'
env.env.update_target(restore_sim=True)

start_time = time.time()
for ep in range(n_eps):
    print("Ep {} of {}".format(ep, n_eps))
    
    for _cstep in range(env.spec.max_episode_steps):
        if GENERATE_VIDEO and (ep in range(GENERATE_VIDEO_EPS) or ep in range(n_eps-GENERATE_VIDEO_EPS, n_eps)):
            frame = env.sim.renderer.render_offscreen(width=400, height=400, camera_id=0)
            
            # Add text overlay
            _current_time = (ep*env.spec.max_episode_steps + _cstep)*env.dt
            frame = np.array(add_text_to_frame(frame,
                    f"t={str(int(_current_time//60)).zfill(2)}:{str(int(_current_time%60)).zfill(2)}min",
                    pos=(285, 3), color=(0, 0, 0), fontsize=18))
            
            frames.append(frame)
        o = env.get_obs()
        a = pi.get_action(o)[0]
        next_o, r, done, ifo = env.step(a) # take an action based on the current observation

        data_store.append({"action":a.copy(), 
                            "jpos":env.sim.data.qpos.copy(), 
                            "mlen":env.sim.data.actuator_length.copy(), 
                            "act":env.sim.data.act.copy(),
                            "reward":r,
                            "solved":env.rwd_dict['solved'].item(),
                            "pose_err":env.get_obs_dict(env.sim)["pose_err"],
                            "MA":env.muscle_fatigue.MA.copy(),
                            "MR":env.muscle_fatigue.MR.copy(),
                            "MF":env.muscle_fatigue.MF.copy(),
                            "ctrl":env.last_ctrl.copy()})
env.close()

## OPTIONALLY: Stored simulated data
if STORE_DATA:
    os.makedirs(f"{env_name}/logs", exist_ok=True)
    np.save(f"{env_name}/logs/fatitest.npy", data_store)

## OPTIONALLY: Render video
if GENERATE_VIDEO:
    os.makedirs(f'{env_name}/videos', exist_ok=True)
    # make a local copy
    skvideo.io.vwrite(f'{env_name}/videos/fatitest.mp4', np.asarray(frames),inputdict={'-r': str(int(1/env.dt))},outputdict={"-pix_fmt": "yuv420p"})

end_time = time.time()
print(f"DURATION: {end_time - start_time:.2f}s")

if GENERATE_VIDEO:
    display(show_video(f'{env_name}/videos/fatitest.mp4'))

In [ ]:
env_name = "myoFatiElbowPose1D6MRandom-v0"

####################

env_test = gym.make(env_name, normalize_act=False)
muscle_names = [env_test.sim.model.id2name(i, "actuator") for i in range(env_test.sim.model.nu) if env_test.sim.model.actuator_dyntype[i] == mujoco.mjtDyn.mjDYN_MUSCLE]
_env_dt = env_test.dt  #0.02

data_store = np.load(f"{env_name}/logs/fatitest.npy", allow_pickle=True)

plt.figure()
for _muscleid in range(len(data_store[0]['MF'])):
    plt.plot(_env_dt*np.arange(len(data_store)), np.array([d['MF'][_muscleid] for d in data_store]), label=muscle_names[_muscleid])
plt.legend()
plt.title('Fatigued Motor Units')

plt.figure()
for _muscleid in range(len(data_store[0]['MR'])):
    plt.plot(_env_dt*np.arange(len(data_store)), np.array([d['MR'][_muscleid] for d in data_store]), label=muscle_names[_muscleid])
plt.legend()
plt.title('Resting Motor Units')

plt.figure()
for _muscleid in range(len(data_store[0]['MA'])):
    plt.plot(_env_dt*np.arange(len(data_store)), np.array([d['MA'][_muscleid] for d in data_store]), label=muscle_names[_muscleid])
plt.legend()
plt.title('Active Motor Units')

plt.figure()
plt.plot(_env_dt*np.arange(len(data_store)), np.array([np.linalg.norm(d['pose_err']) for d in data_store])), plt.title('Pose Error')

plt.figure()
plt.plot(_env_dt*np.arange(len(data_store)), np.array([d['reward'] for d in data_store])), plt.title(f"Reward (Total: {np.array([d['reward'] for d in data_store]).sum():.2f})")

if "solved" in data_store[0]:
    plt.figure()
    plt.scatter(_env_dt*np.arange(len(data_store))[np.array([d['solved'] for d in data_store])], np.array([d['solved'] for d in data_store])[np.array([d['solved'] for d in data_store])]), plt.title(f"Success")

print(f"Muscle Fatigue Equilibrium: {data_store[-1]['MF']}")

Comparison: Policy trained without fatigue

In [ ]:
env_name = "myoFatiElbowPose1D6MRandom-v0"

GENERATE_VIDEO = True
GENERATE_VIDEO_EPS = 4  #number of episodes that are rendered BOTH at the beginning (i.e., without fatigue) and at the end (i.e., with fatigue)

STORE_DATA = True  #store collected data from evaluation run in .npy file
n_eps = 250

###################################

env = gym.make(env_name)

policy = "../../../myosuite/agents/baslines_NPG/myoElbowPose1D6MRandom-v0/2022-02-26_21-16-27/33_env=myoElbowPose1D6MRandom-v0,seed=1/iterations/best_policy.pickle"
pi = pickle.load(open(policy, 'rb'))

env.set_fatigue_reset_random(False)
env.reset(fatigue_reset=True)  #ensure that fatigue is reset before the simulation starts

env.sim.model.cam_poscom0[0]= np.array([-1.3955, -0.3287,  0.6579])

data_store = []
if GENERATE_VIDEO:
    frames = []

env.env.target_jnt_value = env.target_jnt_range[:, 1]
env.env.target_type = 'fixed'
env.env.update_target(restore_sim=True)

start_time = time.time()
for ep in range(n_eps):
    print("Ep {} of {}".format(ep, n_eps))

    for _cstep in range(env.spec.max_episode_steps):
        if GENERATE_VIDEO and (ep in range(GENERATE_VIDEO_EPS) or ep in range(n_eps-GENERATE_VIDEO_EPS, n_eps)):
            frame = env.sim.renderer.render_offscreen(width=400, height=400, camera_id=0)
            
            # Add text overlay
            _current_time = (ep*env.spec.max_episode_steps + _cstep)*env.dt
            frame = np.array(add_text_to_frame(frame,
                    f"t={str(int(_current_time//60)).zfill(2)}:{str(int(_current_time%60)).zfill(2)}min",
                    pos=(285, 3), color=(0, 0, 0), fontsize=18))
            
            frames.append(frame)
        o = env.get_obs()
        a = pi.get_action(o)[0]
        next_o, r, done, ifo = env.step(a) # take an action based on the current observation

        data_store.append({"action":a.copy(), 
                            "jpos":env.sim.data.qpos.copy(), 
                            "mlen":env.sim.data.actuator_length.copy(), 
                            "act":env.sim.data.act.copy(),
                            "reward":r,
                            "solved":env.rwd_dict['solved'].item(),
                            "pose_err":env.get_obs_dict(env.sim)["pose_err"],
                            "MA":env.muscle_fatigue.MA.copy(),
                            "MR":env.muscle_fatigue.MR.copy(),
                            "MF":env.muscle_fatigue.MF.copy(),
                            "ctrl":env.last_ctrl.copy()})
env.close()

## OPTIONALLY: Stored simulated data
if STORE_DATA:
    os.makedirs(f"{env_name}/logs", exist_ok=True)
    np.save(f"{env_name}/logs/fatitest_trained_wo_fatigue.npy", data_store)

## OPTIONALLY: Render video
if GENERATE_VIDEO:
    os.makedirs(f'{env_name}/videos', exist_ok=True)
    # make a local copy
    skvideo.io.vwrite(f'{env_name}/videos/fatitest_trained_wo_fatigue.mp4', np.asarray(frames),inputdict={'-r': str(int(1/env.dt))},outputdict={"-pix_fmt": "yuv420p"})

end_time = time.time()
print(f"DURATION: {end_time - start_time:.2f}s")

if GENERATE_VIDEO:
    display(show_video(f'{env_name}/videos/fatitest_trained_wo_fatigue.mp4'))

In [ ]:
env_name = "myoFatiElbowPose1D6MRandom-v0"

####################

env_test = gym.make(env_name, normalize_act=False)
muscle_names = [env_test.sim.model.id2name(i, "actuator") for i in range(env_test.sim.model.nu) if env_test.sim.model.actuator_dyntype[i] == mujoco.mjtDyn.mjDYN_MUSCLE]
_env_dt = env_test.dt  #0.02

data_store = np.load(f"{env_name}/logs/fatitest_trained_wo_fatigue.npy", allow_pickle=True)

plt.figure()
for _muscleid in range(len(data_store[0]['MF'])):
    plt.plot(_env_dt*np.arange(len(data_store)), np.array([d['MF'][_muscleid] for d in data_store]), label=muscle_names[_muscleid])
plt.legend()
plt.title('Fatigued Motor Units')

plt.figure()
for _muscleid in range(len(data_store[0]['MR'])):
    plt.plot(_env_dt*np.arange(len(data_store)), np.array([d['MR'][_muscleid] for d in data_store]), label=muscle_names[_muscleid])
plt.legend()
plt.title('Resting Motor Units')

plt.figure()
for _muscleid in range(len(data_store[0]['MA'])):
    plt.plot(_env_dt*np.arange(len(data_store)), np.array([d['MA'][_muscleid] for d in data_store]), label=muscle_names[_muscleid])
plt.legend()
plt.title('Active Motor Units')

plt.figure()
plt.plot(_env_dt*np.arange(len(data_store)), np.array([np.linalg.norm(d['pose_err']) for d in data_store])), plt.title('Pose Error')

plt.figure()
plt.plot(_env_dt*np.arange(len(data_store)), np.array([d['reward'] for d in data_store])), plt.title(f"Reward (Total: {np.array([d['reward'] for d in data_store]).sum():.2f})")

if "solved" in data_store[0]:
    plt.figure()
    plt.scatter(_env_dt*np.arange(len(data_store))[np.array([d['solved'] for d in data_store])], np.array([d['solved'] for d in data_store])[np.array([d['solved'] for d in data_store])]), plt.title(f"Success")

print(f"Muscle Fatigue Equilibrium: {data_store[-1]['MF']}")

### Example 2: Fatigue + Resting Period

In this variant, the same target needs to be reached for 5 minutes, followed by a resting period of 2:30 minutes, and another 2:30 minutes of the same task.

In [ ]:
env_name = "myoFatiElbowPose1D6MRandom-v0"

GENERATE_VIDEO = True
GENERATE_VIDEO_EPS = 300  #number of episodes that are rendered BOTH at the beginning (i.e., without fatigue) and at the end (i.e., with fatigue)

STORE_DATA = True  #store collected data from evaluation run in .npy file
n_eps = 300

###################################

env = gym.make(env_name)

policy = f"{env_name}/iterations/best_policy.pickle"
pi = pickle.load(open(policy, 'rb'))

env.set_fatigue_reset_random(False)
env.reset(fatigue_reset=True)  #ensure that fatigue is reset before the simulation starts

env.sim.model.cam_poscom0[0]= np.array([-1.3955, -0.3287,  0.6579])

data_store = []
if GENERATE_VIDEO:
    frames = []

env.env.target_jnt_value = env.target_jnt_range[:, 1]
env.env.target_type = 'fixed'
env.env.update_target(restore_sim=True)

start_time = time.time()
for ep in range(n_eps):
    print("Ep {} of {}".format(ep, n_eps))
    
    for _cstep in range(env.spec.max_episode_steps):
        if GENERATE_VIDEO and (ep in range(GENERATE_VIDEO_EPS) or ep in range(n_eps-GENERATE_VIDEO_EPS, n_eps)):
            frame = env.sim.renderer.render_offscreen(width=480, height=480, camera_id=0)
            
            # Add text overlay
            _current_time = (ep*env.spec.max_episode_steps + _cstep)*env.dt
            frame = np.array(add_text_to_frame(frame,
                    f"t={str(int(_current_time//60)).zfill(2)}:{str(int(_current_time%60)).zfill(2)}min",
                    pos=(365, 3), color=(0, 0, 0), fontsize=18))
            
            if ep >= n_eps*0.5 and ep < n_eps*0.75:
                frame = np.array(add_text_to_frame(frame,
                    f"Resting Phase",
                    pos=(320, 450), color=(84, 184, 81), fontsize=18))

            frames.append(frame)
        o = env.get_obs()
        a = pi.get_action(o)[0]

        if ep >= n_eps*0.5 and ep < n_eps*0.75:
            a[:] = -100000  #resting period (corresponds to zero muscle activations)
            env.sim.model.site_rgba[env.target_sids[0]][-1] = 0  #hide target during resting period
            env.sim.model.tendon_rgba[-1][-1] = 0  #hide error line during resting period
        else:
            env.sim.model.site_rgba[env.target_sids[0]][-1] = 0.2  #visualise target during task
            env.sim.model.tendon_rgba[-1][-1] = 0.2  #visualise error line during task

        next_o, r, done, ifo = env.step(a) # take an action based on the current observation

        data_store.append({"action":a.copy(), 
                            "jpos":env.sim.data.qpos.copy(), 
                            "mlen":env.sim.data.actuator_length.copy(), 
                            "act":env.sim.data.act.copy(),
                            "reward":r,
                            "solved":env.rwd_dict['solved'].item(),
                            "pose_err":env.get_obs_dict(env.sim)["pose_err"],
                            "MA":env.muscle_fatigue.MA.copy(),
                            "MR":env.muscle_fatigue.MR.copy(),
                            "MF":env.muscle_fatigue.MF.copy(),
                            "ctrl":env.last_ctrl.copy()})
env.close()

## OPTIONALLY: Stored simulated data
if STORE_DATA:
    os.makedirs(f"{env_name}/logs", exist_ok=True)
    np.save(f"{env_name}/logs/fatitest_recovery.npy", data_store)

## OPTIONALLY: Render video
if GENERATE_VIDEO:
    os.makedirs(f'{env_name}/videos', exist_ok=True)
    # make a local copy
    skvideo.io.vwrite(f'{env_name}/videos/fatitest_recovery.mp4', np.asarray(frames),inputdict={'-r': str(int(1/env.dt))},outputdict={"-pix_fmt": "yuv420p"})

end_time = time.time()
print(f"DURATION: {end_time - start_time:.2f}s")

if GENERATE_VIDEO:
    display(show_video(f'{env_name}/videos/fatitest_recovery.mp4'))

In [ ]:
env_name = "myoFatiElbowPose1D6MRandom-v0"

####################

env_test = gym.make(env_name, normalize_act=False)
muscle_names = [env_test.sim.model.id2name(i, "actuator") for i in range(env_test.sim.model.nu) if env_test.sim.model.actuator_dyntype[i] == mujoco.mjtDyn.mjDYN_MUSCLE]
_env_dt = env_test.dt  #0.02

data_store = np.load(f"{env_name}/logs/fatitest_recovery.npy", allow_pickle=True)

plt.figure()
for _muscleid in range(len(data_store[0]['MF'])):
    plt.plot(_env_dt*np.arange(len(data_store)), np.array([d['MF'][_muscleid] for d in data_store]), label=muscle_names[_muscleid])
plt.legend()
plt.title('Fatigued Motor Units')

plt.figure()
for _muscleid in range(len(data_store[0]['MR'])):
    plt.plot(_env_dt*np.arange(len(data_store)), np.array([d['MR'][_muscleid] for d in data_store]), label=muscle_names[_muscleid])
plt.legend()
plt.title('Resting Motor Units')

plt.figure()
for _muscleid in range(len(data_store[0]['MA'])):
    plt.plot(_env_dt*np.arange(len(data_store)), np.array([d['MA'][_muscleid] for d in data_store]), label=muscle_names[_muscleid])
plt.legend()
plt.title('Active Motor Units')

plt.figure()
plt.plot(_env_dt*np.arange(len(data_store)), np.array([np.linalg.norm(d['pose_err']) for d in data_store])), plt.title('Pose Error')

plt.figure()
plt.plot(_env_dt*np.arange(len(data_store)), np.array([d['reward'] for d in data_store])), plt.title(f"Reward (Total: {np.array([d['reward'] for d in data_store]).sum():.2f})")

if "solved" in data_store[0]:
    plt.figure()
    plt.scatter(_env_dt*np.arange(len(data_store))[np.array([d['solved'] for d in data_store])], np.array([d['solved'] for d in data_store])[np.array([d['solved'] for d in data_store])]), plt.title(f"Success")

print(f"Muscle Fatigue Equilibrium: {data_store[-1]['MF']}")